# GenAI Pipeline — Patents

LLM-based screening of patents for ingredient classification.
Uses Claude with structured output via the Anthropic Python SDK.

Adapted from the publications GenAI pipeline.

### 1. Imports and Configuration

In [1]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator, create_model
from typing import Literal

load_dotenv("../../.env")

DB_PATH = "../../patents_training.db"
OUTPUT_DIR = Path(".")

### 2. Data Inspection

In [4]:
con = duckdb.connect(DB_PATH, read_only=True)
print("Tables:")
print(con.sql("SHOW TABLES").df())

df = con.sql("SELECT * FROM patents_raw").df()   # ← UPDATE: table name
con.close()

df = df[df['scope'] == 'in']

print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nScope distribution:")
print(df["scope"].value_counts())
print(f"\nIngredient distribution:")
print(df["ingredient"].value_counts())
df.head()

Tables:
                 name
0  patents_embeddings
1         patents_raw

Shape: (1275, 15)

Columns: ['id', 'family_id', 'application_number', 'title', 'abstract', 'cpc', 'publication_year', 'jurisdiction', 'scope', 'pillar', 'subpillar', 'research_category', 'endproduct', 'ingredient', 'truncated']

Scope distribution:
scope
in    1275
Name: count, dtype: int64

Ingredient distribution:
ingredient
Isolates, concentrates, and flours    112
Emulsions, gels, and binders           92
Flavours and aromas                    80
Fats and oils                          44
Colours                                18
Name: count, dtype: int64


,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
2,US-20170298457-A1,42829610,US15360298,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12N9/1205', 'A23C2220/206', ...",2017,US,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
3,EP-2473058-A1,42829610,EP10751643A,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12R2001/46', 'C12Y207/01006'...",2012,EP,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
28,EP-4424172-A3,52462090,EP24182765.8,A PROTEINACEOUS MEAT ANALOGUE HAVING AN IMPROV...,The invention concerns an extended shelf-life ...,"['A23V2002/00', 'A23V2200/20', 'A23J3/22', 'A2...",2025,EP,in,PB,NaN,End product formulation,Meat,NaN,False
29,US-11666069-B2,52462090,US17530304,Proteinaceous meat analogue having an improved...,An extended shelf-life proteinaceous meat anal...,"['A23J3/22', 'A23J3/18', 'A23V2200/20', 'A23V2...",2023,US,in,PB,NaN,End product formulation,Meat,NaN,False
32,EP-4627930-A3,52596486,EP25169128.3,VARIANTS OF CHYMOSIN WITH IMPROVED MILK-CLOTTI...,Variants of chymosin with improved milk-clotti...,"['A23C19/04', 'A23C19/041', 'C12N9/6483', 'A23...",2026,EP,in,PB,NaN,End product formulation,Cheese,NaN,False


### 3. Balanced Subset Creation

Two test sets with balanced representation across scope and pillar:
- `initial_test_data`: ~14 records (6 out, 2 PB, 2 F, 2 cultivated, 2 cross-cutting)
- `test_data_100`: ~110 records (40 out, 30 PB, 15 F, 15 cultivated, 10 cross-cutting)

Adjust counts to match the actual distribution in your patents dataset.

In [5]:
RANDOM_STATE = 4

def create_balanced_sample(df, category_counts, category_col='ingredient', random_state=RANDOM_STATE):
    """
    category_counts: dict mapping ingredient category -> n, e.g.
        {"Flavours and aromas": 25, "N/A": 25}
    Categories with no matching rows are skipped with a warning.
    If n exceeds available rows, all available rows are taken (with a warning).
    """
    samples = []
    for cat, n in category_counts.items():
        subset = df[df[category_col] == cat]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        if n > available:
            print(f"  Warning: '{cat}' — requested {n} but only {available} available, taking all.")
            n = available
        samples.append(subset.sample(n=n, random_state=random_state))
    combined = pd.concat(samples, ignore_index=True)
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)


In [6]:
# Fill NaN ingredient values with 'N/A' for sampling purposes
df_ing = df.copy()
df_ing['ingredient'] = df_ing['ingredient'].fillna('N/A')

test_data = create_balanced_sample(df_ing, {
    'N/A': 25,
    'Isolates, concentrates, and flours': 10,
    'Emulsions, gels, and binders': 10,
    'Flavours and aromas': 10,
    'Fats and oils': 10,
    'Colours': 10,
}, random_state=RANDOM_STATE)
print(f'test_data: {test_data.shape}')


test_data: (75, 15)


### 4. Save Subsets to Excel

In [7]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=OUTPUT_DIR):
    path = output_dir / filename
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} records to {path}")

#save_subset(test_data, f"ingredient_test_data_rand{RANDOM_STATE}.csv")

Saved 75 records to ingredient_test_data_rand4.csv


In [22]:
# Read subset data from file
test_data = pd.read_csv(f"ingredient_test_data_rand{RANDOM_STATE}.csv")


In [25]:
test_data.value_counts('ingredient')


ingredient
Isolates, concentrates, and flours    10
Emulsions, gels, and binders          10
Colours                               10
Flavours and aromas                    9
Fats and oils                          9
Name: count, dtype: int64

### 5. Load Prompt and Select Dataset

In [36]:
PROMPT_PATH = "prompt_ingredient_patents_work.md"   # ← UPDATE: path to current prompt version
PROMPT_VERSION = "v3"

# ← CHANGE THIS to switch between datasets
DATASET = test_data
DATASET_STR = "test_data"


In [37]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)

You are an expert in alternative proteins and food technology.

Your task is to classify a patent on alternative proteins by the type of food ingredient it primarily describes, based on its title and abstract.

IMPORTANT: The majority of alternative protein patents do NOT describe a standalone food ingredient. They describe complete food products (meat analogues, dairy alternatives, desserts), production or processing methods, microbial organisms or strains, bioreactors, or equipment. Assign N/A for all of these. Only assign an ingredient type label when the patent clearly and primarily describes a standalone ingredient — a substance intended to be added to or incorporated into food products.

Before assigning a non-N/A label, ask: "Is this patent primarily about a standalone FOOD INGREDIENT — a substance that would be added to or incorporated into other food products?" If yes, assign the appropriate ingredient type. If no, assign N/A.

Key decision rules:
- A protein isolate, concentr

### 6. API Call with Structured Output

In [38]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model
# If using Opus, comment out TEMPERATURE below

MAX_TOKENS = 512         # max tokens in response
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0
RETRY_MAX_SECONDS = 90.0

REPETITIONS = 1          # number of full runs; increase to measure output variance

# ================================================================
# CHECKPOINT CONFIG
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

In [39]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

INGREDIENT_CATS = [
    "Isolates, concentrates, and flours",
    "Emulsions, gels, and binders",
    "N/A",
    "Flavours and aromas",
    "Colours",
    "Fats and oils",
]

def make_schema(cats, include_reasoning):
    cats_map = {c.lower(): c for c in cats}
    cat_type = Literal[*cats]

    class _Base(BaseModel):
        @field_validator("primary", "secondary", mode="before", check_fields=False)
        @classmethod
        def normalise_case(cls, v):
            if isinstance(v, str):
                return cats_map.get(v.lower(), v)
            return v

    fields = {"primary": (cat_type, ...), "secondary": (cat_type, ...)}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    return create_model("ClassificationSchema", __base__=_Base, **fields)

ClassificationSchema = make_schema(INGREDIENT_CATS, INCLUDE_REASONING)
print(f"Schema built: {INGREDIENT_CATS}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_patent(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output


Schema built: ['Isolates, concentrates, and flours', 'Emulsions, gels, and binders', 'N/A', 'Flavours and aromas', 'Colours', 'Fats and oils']


### 7. Error Handling with Retry

In [40]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter

In [41]:
def classify_with_error_handling(row, system_prompt):
    pat_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_patent(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {pat_id}: model returned no structured output")
                return {"id": pat_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()}
            output["id"] = pat_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pat_id}: {last_error}")
    return {"id": pat_id, "status": "api_error", "error": str(last_error)}

### 8. Checkpoint Helpers

In [42]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()

### 9. Run on Test Data

In [43]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df


Run 1 / 1
  [1/75] US-20240407394-A1
  [2/75] CL-2026000647-A1
  [3/75] EP-4633376-A1
  [4/75] WO-2025099060-A1
  [5/75] US-20250212923-A1
  [6/75] EP-4665157-A1
  [7/75] CN-122121753-A
  [8/75] CN-119365088-A
  [9/75] WO-2025078458-A3
  [10/75] US-20250049060-A1
  [11/75] WO-2025262697-A1
  [12/75] US-20250380723-A1
  [13/75] AU-2019394578-B2
  [14/75] EP-4476316-A1
  [15/75] WO-2025045796-A1
  [16/75] US-20240407392-A1
  [17/75] EP-3897693-A1
  [18/75] WO-2025132325-A1
  [19/75] US-12545887-B2
  [20/75] WO-2025088115-A1
  [21/75] JP-2025522929-A
  [22/75] CN-120981168-A
  [23/75] US-20210244046-A1
  [24/75] EP-4742909-A1
  [25/75] WO-2025242924-A1
  [26/75] WO-2025132527-A1
  [27/75] SE-547148-C2
  [28/75] WO-2025191050-A1
  [29/75] MX-2026000777-A
  [30/75] EP-4479515-A1
  [31/75] US-20250331537-A1
  [32/75] EP-4398740-A1
  [33/75] US-20220312794-A1
  [34/75] US-20250313784-A1
  [35/75] EP-4531592-A1
  [36/75] EP-3670646-A1
  [37/75] US-20220007693-A1
  [38/75] MX-2026003171-A
  [3

,primary_LLM,secondary_LLM,reasoning_LLM,id,status,run
0,"Isolates, concentrates, and flours","Emulsions, gels, and binders",The method processes aleurone-containing cerea...,US-20240407394-A1,ok,1
1,Flavours and aromas,"Isolates, concentrates, and flours",The patent primarily describes a fermented pla...,CL-2026000647-A1,ok,1
2,N/A,"Emulsions, gels, and binders",The patent primarily describes a complete plan...,EP-4633376-A1,ok,1
3,Flavours and aromas,N/A,The patent explicitly describes a flavor modul...,WO-2025099060-A1,ok,1
4,Flavours and aromas,"Emulsions, gels, and binders",The patent primarily describes a flavour deliv...,US-20250212923-A1,ok,1
...,...,...,...,...,...,...
70,Flavours and aromas,"Isolates, concentrates, and flours",The patent describes a yeast-derived natural e...,FR-3162341-A1,ok,1
71,N/A,"Isolates, concentrates, and flours",This patent describes a device and method for ...,WO-2025195977-A1,ok,1
72,Colours,N/A,The patent explicitly describes food colouring...,EP-4531591-A1,ok,1
73,Fats and oils,N/A,The patent describes adipocyte maturation from...,WO-2025141193-A1,ok,1


In [47]:
result_cols = ["id", "run", "primary_LLM", "secondary_LLM", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "pillar", "ingredient"]].merge(
    results_df[result_cols], on="id", how="left"
)

# NaN ingredient in GT corresponds to 'N/A' in LLM output
# Fill NaN with 'N/A' for comparison
comparison["ingredient_gt"] = comparison["ingredient"].fillna("N/A")

comparison["primary_correct"] = comparison["ingredient_gt"] == comparison["primary_LLM"]
comparison["either_correct"]  = (
    (comparison["ingredient_gt"] == comparison["primary_LLM"]) |
    (comparison["ingredient_gt"] == comparison["secondary_LLM"])
)

# Overall metrics
n = len(comparison)
print(f"Top-1 accuracy (primary match):  {comparison['primary_correct'].mean():.0%}  (n={n})")
print(f"Top-2 accuracy (either match):   {comparison['either_correct'].mean():.0%}  (n={n})")

# Per-category breakdown
cat_stats = (
    comparison.groupby("ingredient_gt")
    .agg(
        n=("primary_correct", "count"),
        primary_correct=("primary_correct", "sum"),
        top2_correct=("either_correct", "sum"),
    )
    .assign(
        primary_acc=lambda d: (d["primary_correct"] / d["n"]).map("{:.0%}".format),
        top2_acc=lambda d: (d["top2_correct"] / d["n"]).map("{:.0%}".format),
    )
)
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "pillar", "ingredient_gt",
                "primary_LLM", "secondary_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["primary_correct", "either_correct"]
comparison[display_cols]


Top-1 accuracy (primary match):  91%  (n=75)
Top-2 accuracy (either match):   97%  (n=75)


,n,primary_correct,top2_correct,primary_acc,top2_acc
ingredient_gt,,,,,
Colours,10,10,10,100%,100%
"Emulsions, gels, and binders",10,8,9,80%,90%
Fats and oils,9,8,9,89%,100%
Flavours and aromas,9,9,9,100%,100%
"Isolates, concentrates, and flours",10,8,10,80%,100%
N/A,27,25,26,93%,96%


,id,title,abstract,pillar,ingredient_gt,primary_LLM,secondary_LLM,reasoning_LLM,primary_correct,either_correct
0,US-20240407394-A1,METHOD FOR PROCESSING CEREAL GRAIN,The invention pertains to a method for process...,PB,"Isolates, concentrates, and flours","Isolates, concentrates, and flours","Emulsions, gels, and binders",The method processes aleurone-containing cerea...,True,True
1,CL-2026000647-A1,Method for preparing fermented plant-based pro...,The invention relates to a method for manufact...,PB,Flavours and aromas,Flavours and aromas,"Isolates, concentrates, and flours",The patent primarily describes a fermented pla...,True,True
2,EP-4633376-A1,PLANT-BASED MILK COMPRISING SUNFLOWER OLEOSOMES,The present invention relates to a plant-based...,PB,"Emulsions, gels, and binders",N/A,"Emulsions, gels, and binders",The patent primarily describes a complete plan...,False,True
3,WO-2025099060-A1,FLAVOR MODULATING COMPOSITIONS,A flavor modulating composition for improving ...,PB,Flavours and aromas,Flavours and aromas,N/A,The patent explicitly describes a flavor modul...,True,True
4,US-20250212923-A1,FLAVOUR DELIVERY SYSTEM,The present invention provides a solid or semi...,PB,Flavours and aromas,Flavours and aromas,"Emulsions, gels, and binders",The patent primarily describes a flavour deliv...,True,True
...,...,...,...,...,...,...,...,...,...,...
70,FR-3162341-A1,NEW GABA-RICH MASK PRODUCT,The present invention relates to a natural ext...,F,Flavours and aromas,Flavours and aromas,"Isolates, concentrates, and flours",The patent describes a yeast-derived natural e...,True,True
71,WO-2025195977-A1,DEVICE AND METHOD FOR PRODUCING STRUCTURED FOO...,The invention relates to a device and a method...,PB,N/A,N/A,"Isolates, concentrates, and flours",This patent describes a device and method for ...,True,True
72,EP-4531591-A1,COLOURING COMPOSITIONS,The invention relates to novel food colouring ...,PB,Colours,Colours,N/A,The patent explicitly describes food colouring...,True,True
73,WO-2025141193-A1,ADIPOCYTE MATURATION,The present invention relates to pluripotent s...,CM,Fats and oils,Fats and oils,N/A,The patent describes adipocyte maturation from...,True,True


### 10. Save to Excel for Prompt Debugging

Order of working:
1. Create a new version folder in `1_prompt_debugging/`.
2. Copy in the previous prompt, label with the new version number, make updates.
3. Edit step 10 output directory and step 5 prompt/dataset selection.
4. Run steps 5–10.
5. Manually review results.
6. Document what changed and why. Repeat from step 1.

In [45]:
save_dir = Path(f"{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "top1_accuracy", "value": f"{comparison['primary_correct'].mean():.0%}", "n": n},
    {"metric": "top2_accuracy", "value": f"{comparison['either_correct'].mean():.0%}",  "n": n},
])

out_path = save_dir / f"{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")


Saved to v3\v3_claude-sonnet-4-6_results.xlsx


In [46]:
# Create dataset of only incorrectly classified rows, for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["primary_correct"], "id"]
incorrect_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_data


,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,EP-4633376-A1,84568997,EP23822247.5,PLANT-BASED MILK COMPRISING SUNFLOWER OLEOSOMES,The present invention relates to a plant-based...,['A23C11/10'],2025,EP,in,PB,NaN,End product formulation,Milk and milk proteins,"Emulsions, gels, and binders",False
1,WO-2025078458-A3,88373901,EP2024/078433,NOVEL FOOD COMPOSITION AND METHOD FOR PRODUCTI...,The present disclosure relates to a process fo...,"['C12Y302/01', 'C12C5/006', 'A23G1/34', 'C12P1...",2026,WO,in,F,PF,Ingredient optimisation,Milk and milk proteins,"Emulsions, gels, and binders",False
2,US-20210244046-A1,62814969,US17251079,NON-VITAL WHEAT PROTEIN AND ITS PRODUCTION PRO...,This invention relates to a non-vital wheat pr...,"['C07K14/415', 'C07K1/34', 'B01D61/149', 'B01D...",2021,US,in,PB,NaN,Ingredient optimisation,Agnostic,"Isolates, concentrates, and flours",False
3,SE-547148-C2,91956344,SE2330037A,Method of Preparing a Food Component from Pulses,A method of preparing a food component from pu...,"['A23J3/346', 'A23L11/50', 'A23L11/60', 'A23J3...",2025,SE,in,PB,NaN,Ingredient optimisation,Milk and milk proteins,NaN,False
4,MX-2026000777-A,87556283,MX2026000777A,METHOD FOR PRODUCING CASEIN AND USES THEREOF,The invention relates to new methods for produ...,"['A23J1/008', 'A23C20/02', 'C12N15/70', 'C07K1...",2026,MX,in,F,PF,Target molecule selection,Cheese,NaN,False
5,US-12550910-B2,69469005,US17761762,Semi-finished powdery food product based on ve...,Disclosed is a powder and dehydrated product e...,"['A23P10/40', 'A23G9/327', 'A23L19/01', 'A23L9...",2026,US,in,PB,NaN,Ingredient optimisation,Milk and milk proteins,"Isolates, concentrates, and flours",False
6,US-20260060266-A1,83151825,US19106889,CO-EXTRACTION METHOD FOR PREPARING A STABLE OI...,The invention relates to a method of preparing...,"['A23D9/05', 'C11B1/10', 'C11B1/04', 'A23D9/02']",2026,US,in,PB,NaN,Ingredient optimisation,Cross-cutting,Fats and oils,False
